In [4]:
!pip install pandas scikit-learn imbalanced-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 590.7 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.4/235.4 kB 658.2 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 378.5 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 172.4 kB/s eta 0:00:0000:0100:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 166.3 kB/s eta 0:00:0000:0100:07
  Attempting uninstall: numpy
    Found existing installation: numpy 1.24.4
    Uninstalling numpy-1.24.4:
      Successfully uninstalled numpy-1.24.4
  Attempting uninstall: scipy
    Found existing installation: scipy 1.11.3
    Uninstalling scipy-1.11.3:
      Successfully uninstalled scipy-1.11.3
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.3.1
    Uninstalling scikit-learn-1.3.1:
      Successfully uninstalled scikit-learn-

In [5]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from imblearn.over_sampling import SMOTE

# --- Configuration & Path Handling ---
# Using a relative path to avoid the /home/student permission issues
DATA_DIR = './data'
DATA_PATH = os.path.join(DATA_DIR, 'network_flow_features.csv')
os.makedirs(DATA_DIR, exist_ok=True)

print("🛡️ Intrusion Detection Lab Initialized")

# --- Step 1: Data Preparation ---
if not os.path.exists(DATA_PATH):
    print(f"Dataset not found. Generating a synthetic cybersecurity dataset...")
    # Creating a dummy dataset: 1000 flows, 5 features, imbalanced (95% Normal, 5% Attack)
    np.random.seed(42)
    data = {
        'avg_packet_size': np.random.normal(500, 100, 1000),
        'flow_duration': np.random.normal(10, 2, 1000),
        'src_port': np.random.randint(1024, 65535, 1000),
        'dst_port': np.random.randint(20, 1000, 1000),
        'payload_entropy': np.random.uniform(3, 5, 1000),
        'label': ['Normal'] * 950 + ['Attack'] * 50
    }
    df = pd.DataFrame(data)
    # Make 'Attack' traffic look different (higher entropy and duration)
    df.loc[df['label'] == 'Attack', 'payload_entropy'] += 2.5
    df.loc[df['label'] == 'Attack', 'flow_duration'] *= 5
    df.to_csv(DATA_PATH, index=False)
    print(f"Created dataset at {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} network flows.")

# --- Step 2: Feature Engineering & Label Encoding ---
X = df.drop('label', axis=1)
y = df['label']

# Convert 'Normal'/'Attack' to 0/1
label_mapping = {label: idx for idx, label in enumerate(y.unique())}
y_numeric = y.map(label_mapping)
print(f"Label Mapping: {label_mapping}")

# --- Step 3: Handling Data Imbalance (SMOTE) ---
# In cybersecurity, attacks are the 'needle in the haystack.'
# SMOTE creates synthetic needles so the model learns their shape.
X_train, X_test, y_train, y_test = train_test_split(
    X, y_numeric, test_size=0.3, random_state=42, stratify=y_numeric
)

print(f"Pre-SMOTE 'Attack' count: {sum(y_train == label_mapping['Attack'])}")

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f"Post-SMOTE 'Attack' count: {sum(y_train_res == label_mapping['Attack'])}")

# --- Step 4: Training the Random Forest ---
# Using 100 trees. class_weight='balanced' is a backup for the SMOTE process.
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
print("\nTraining Random Forest Classifier...")
rf_model.fit(X_train_res, y_train_res)
print("Training Complete.")

# --- Step 5: Evaluation ---
y_pred = rf_model.predict(X_test)

print("\n" + "="*30)
print("   PERFORMANCE EVALUATION")
print("="*30)
print(f"Overall Accuracy: {accuracy_score(y_test, y_pred):.4f}")

# Reverse mapping for the report
reverse_mapping = {v: k for k, v in label_mapping.items()}
y_test_labels = [reverse_mapping[i] for i in y_test]
y_pred_labels = [reverse_mapping[i] for i in y_pred]

print("\nClassification Report:")
print(classification_report(y_test_labels, y_pred_labels))

# Feature Importance - The "Why"
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
print("\nTop Predictors of Malicious Traffic:")
print(importances.sort_values(ascending=False))

ImportError: cannot import name 'tarfile_extractall' from 'sklearn.utils.fixes' (/opt/conda/lib/python3.11/site-packages/sklearn/utils/fixes.py)

In [1]:
# for next task, integration with ids

In [6]:
import joblib

# Save the model and the label mapping for the deployment script
model_data = {
    'model': rf_model,
    'mapping': label_mapping,
    'features': X.columns.tolist()
}
joblib.dump(model_data, 'ids_random_forest.joblib')
print("Model exported successfully!")

NameError: name 'rf_model' is not defined